# Dry-run trading analysis

Explores `data/dry_run_transactions.jsonl`: **2-day vs 7-day** markets (same ≤72h rule as `features._classify_period`), **cash balance**, **realized P&L**, and **trade-level patterns**.

Reuses helpers from `visualize_dry_run.py` for loading and period typing.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from visualize_dry_run import (
    fig_balance_timeline,
    fig_comparison,
    load_transactions,
    summarize_by_type,
)

DATA = Path("data/dry_run_transactions.jsonl")
df = load_transactions(DATA)
summary = summarize_by_type(df)

buys = df[df["action"] == "buy"]
sells = df[df["action"] == "sell"]

print(f"Rows: {len(df)}  |  Buys: {len(buys)}  |  Sells: {len(sells)}")
print(f"UTC span: {df['time'].min()} → {df['time'].max()}")

Rows: 253  |  Buys: 142  |  Sells: 111
UTC span: 2026-03-18 06:38:22.585448+00:00 → 2026-04-13 16:09:17.091925+00:00


## Snapshot metrics

Cash **balance** is after each row in the log (not mark-to-market for open positions).

In [2]:
start_cash = float(df["balance"].iloc[0]) + float(buys.iloc[0]["usd"]) if len(buys) else float("nan")
end_cash = float(df["balance"].iloc[-1])
total_buy = float(buys["usd"].sum())
realized = float(sells["profit_usd"].sum()) if len(sells) else 0.0
win_rate = float((sells["profit_usd"] > 0).mean()) if len(sells) else float("nan")
median_sell_pnl = float(sells["profit_usd"].median()) if len(sells) else float("nan")

display(
    pd.DataFrame(
        {
            "metric": [
                "Implied start cash (before first buy)",
                "End cash (last log balance)",
                "Δ cash (end − start)",
                "Total $ deployed (sum of buys)",
                "Sum realized P&L (all sells)",
                "Sell win rate (profit_usd > 0)",
                "Median P&L per sell",
            ],
            "value": [
                f"${start_cash:.2f}",
                f"${end_cash:.2f}",
                f"${end_cash - start_cash:+.2f}",
                f"${total_buy:.2f}",
                f"${realized:+.2f}",
                f"{win_rate:.1%}" if win_rate == win_rate else "—",
                f"${median_sell_pnl:+.2f}" if median_sell_pnl == median_sell_pnl else "—",
            ],
        }
    )
)

summary

,metric,value
0,Implied start cash (before first buy),$100.00
1,End cash (last log balance),$153.64
2,Δ cash (end − start),$+53.64
3,Total $ deployed (sum of buys),$142.00
4,Sum realized P&L (all sells),$+66.64
5,Sell win rate (profit_usd > 0),27.9%
6,Median P&L per sell,$-0.84


,period_type,buys_n,buy_usd,sells_n,sell_proceeds,realized_profit_usd,roi_on_buys_pct
0,2day,62,62.0,48,81.106834,24.106834,38.88
1,7day,77,77.0,61,109.781484,40.781484,52.96
2,unknown,3,3.0,2,4.750000,1.750000,58.33


## Insight: 2-day vs 7-day

**2-day** = event window in slug ≤72h (e.g. `march-21-march-23`). **7-day** = longer weekly-style windows.

Realized profit is attributed by **sell** row’s `event_slug` (and buys by buy slug for capital deployed).

In [3]:
sub = summary[summary["period_type"].isin(["2day", "7day"])]
if len(sub) >= 2:
    p2 = float(sub.loc[sub["period_type"] == "2day", "realized_profit_usd"].sum())
    p7 = float(sub.loc[sub["period_type"] == "7day", "realized_profit_usd"].sum())
    b2 = float(sub.loc[sub["period_type"] == "2day", "buy_usd"].sum())
    b7 = float(sub.loc[sub["period_type"] == "7day", "buy_usd"].sum())
    winner = "2-day" if p2 > p7 else "7-day" if p7 > p2 else "tie"
    print(
        f"Realized P&L — 2-day: ${p2:+.2f} (on ${b2:.2f} buys)  |  "
        f"7-day: ${p7:+.2f} (on ${b7:.2f} buys)"
    )
    print(f"Higher realized P&L: {winner}")
else:
    print("Not enough mixed 2-day / 7-day rows for a clean comparison.")

fig_comparison(summary).show()

Realized P&L — 2-day: $+24.11 (on $62.00 buys)  |  7-day: $+40.78 (on $77.00 buys)
Higher realized P&L: 7-day


## Cash balance & cumulative realized P&L

Green: wallet cash after each event. Orange: cumulative sum of `profit_usd` on sells only (closed-trade attribution).

In [4]:
cum_pnl = sells.sort_values("time")["profit_usd"].cumsum()
t_sell = sells.sort_values("time")["time"]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Cash balance (all rows)", "Cumulative realized P&L (sells only)"),
)
for tr in fig_balance_timeline(df).data:
    fig.add_trace(tr, row=1, col=1)
fig.add_trace(
    go.Scatter(
        x=t_sell,
        y=cum_pnl,
        mode="lines",
        name="Cum. realized",
        line=dict(color="#FFA15A", width=2),
    ),
    row=2,
    col=1,
)
fig.update_layout(height=700, template="plotly_white", hovermode="x unified")
fig.update_yaxes(title_text="USD", row=1, col=1)
fig.update_yaxes(title_text="USD", row=2, col=1)
fig.update_xaxes(title_text="Time (UTC)", row=2, col=1)
fig.show()

## Sell-level distribution

How often do exits help or hurt? **Return multiple** is `return_x` (proceeds / cost basis for that exit).

In [5]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("P&L per sell (USD)", "Return multiple (return_x)"),
)
fig.add_trace(
    go.Histogram(x=sells["profit_usd"], nbinsx=30, name="profit_usd", marker_color="#636EFA"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(x=sells["return_x"], nbinsx=30, name="return_x", marker_color="#EF553B"),
    row=1,
    col=2,
)
fig.update_layout(height=400, template="plotly_white", showlegend=False)
fig.show()

print(
    f"Largest win: ${sells['profit_usd'].max():.2f}  |  "
    f"Largest loss: ${sells['profit_usd'].min():.2f}"
)

Largest win: $10.60  |  Largest loss: $-2.63


## P&L by event (slug)

Aggregates **realized** `profit_usd` per `event_slug` on sell rows. Useful to see which markets drove gains or losses.

In [6]:
by_event = (
    sells.assign(short_slug=sells["event_slug"].str.replace("elon-musk-of-tweets-", "", regex=False))
    .groupby("short_slug", as_index=False)
    .agg(sells=("profit_usd", "count"), realized_pnl=("profit_usd", "sum"))
    .sort_values("realized_pnl", ascending=True)
)

fig = px.bar(
    by_event,
    x="realized_pnl",
    y="short_slug",
    orientation="h",
    color="realized_pnl",
    color_continuous_scale="RdYlGn",
    title="Realized P&L by event (short slug)",
    height=max(400, 28 * len(by_event)),
)
fig.update_layout(template="plotly_white", yaxis=dict(categoryorder="total ascending"))
fig.show()

print("Bottom 3 events by realized P&L:")
display(by_event.head(3))
print("Top 3 events by realized P&L:")
display(by_event.tail(3).iloc[::-1])

Bottom 3 events by realized P&L:


,short_slug,sells,realized_pnl
5,april-3-april-10,6,-5.808903
16,march-23-march-25,4,-5.076301
6,april-4-april-6,4,-4.362764


Top 3 events by realized P&L:


,short_slug,sells,realized_pnl
8,april-7-april-14,6,22.154566
10,march-13-march-20,5,16.341667
1,april-11-april-13,4,15.323077


## Activity rhythm

Buys and sells per **calendar day** (UTC).

In [7]:
df["day"] = df["time"].dt.date
daily = (
    df.groupby(["day", "action"], as_index=False)
    .size()
    .pivot(index="day", columns="action", values="size")
    .fillna(0)
    .astype(int)
)
for c in ["buy", "sell"]:
    if c not in daily.columns:
        daily[c] = 0
daily = daily[["buy", "sell"]]

fig = go.Figure(
    data=[
        go.Bar(name="buy", x=daily.index.astype(str), y=daily["buy"], marker_color="#636EFA"),
        go.Bar(name="sell", x=daily.index.astype(str), y=daily["sell"], marker_color="#EF553B"),
    ]
)
fig.update_layout(
    barmode="group",
    title="Trades per day (UTC)",
    xaxis_title="Date",
    yaxis_title="Count",
    template="plotly_white",
    height=420,
)
fig.show()

daily

action,buy,sell
day,,
2026-03-18,12,0
2026-03-19,6,6
2026-03-20,7,6
2026-03-22,4,4
2026-03-23,11,8
2026-03-24,6,5
2026-03-26,7,7
2026-03-27,6,4
2026-03-28,3,1


## Model signal vs entry (buys only)

`pred` is the model’s predicted tweet level at buy time; scatter shows **entry price** vs **pred** (cheap tail vs model conviction).

In [8]:
if "pred" in buys.columns and buys["pred"].notna().any():
    fig = px.scatter(
        buys,
        x="pred",
        y="price",
        color="period_type",
        hover_data=["event_slug", "bracket", "time"],
        title="Buy: model pred vs market price",
        labels={"pred": "Predicted tweets (model)", "price": "Entry price"},
    )
    fig.update_layout(template="plotly_white", height=500)
    fig.show()
else:
    print("No pred column on buys; skip.")

---

### How to read results

- **Realized P&L** only includes closed sells; open positions are not marked in the balance series beyond cash spent/received.
- **2-day vs 7-day** classification comes from parsing dates in `event_slug`, matching the ≤72h cutoff used elsewhere in this repo.
- For a static HTML export of the core dashboard, run: `python visualize_dry_run.py`.